# DeBERTa-v3 on Kaggle (P100 / T4 GPU)

## One-time setup on Kaggle

1. **Account:** [kaggle.com](https://www.kaggle.com) → verify **phone number** (required for **GPU**).
2. **Create a Dataset** — upload **`pids_bench_v3.zip`** and **`project_src.zip`** (or the folders). Kaggle **auto-unzips** uploads, so under **Input** you usually see **folders** `pids_bench_v3` and `project_src`, not `.zip` files. **This notebook supports both.**
3. **New Notebook** → **Add data** → attach that dataset.
4. **Settings:** **GPU** + **Internet ON**.

**Tesla P100:** Kaggle’s default PyTorch does **not** support P100 (`sm_60`). Cell 2 installs **PyTorch + CUDA 11.8** wheels, which do. **Easier alternative:** in session settings, pick **T4** if offered — then the default stack works without the extra install.

---
**Trains DeBERTa only** (same as Colab `RUN_DISTILBERT=False`).

In [ ]:
# ── Cell 1: Locate data + code (zip OR folder — Kaggle often auto-unzips uploads) ──
import os

# Optional: set full paths from the Input tab if auto-detect fails
MANUAL_DATA = None  # e.g. '/kaggle/input/pids-bench/pids_bench_v3'
MANUAL_SRC = None   # e.g. '/kaggle/input/pids-bench/project_src'

ROOT = '/kaggle/input'

def all_zips(root=ROOT):
    out = []
    if not os.path.isdir(root):
        return out
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith('.zip'):
                out.append(os.path.join(r, f))
    return out

def find_named_dir(root, name):
    """Prefer .../dataset_slug/name (Kaggle layout)."""
    name_l = name.lower()
    found = []
    if not os.path.isdir(root):
        return None
    for r, dirs, _ in os.walk(root):
        base = os.path.basename(r)
        if base.lower() == name_l and os.path.isdir(r):
            found.append(r)
    return found[0] if found else None

def pick_data_zip(zips):
    for z in zips:
        b = os.path.basename(z).lower().replace('-', '_')
        if 'pids_bench_v3' in b or (b.startswith('pids') and 'bench' in b and 'v3' in b):
            return z
    return None

def pick_src_zip(zips):
    for z in zips:
        b = os.path.basename(z).lower().replace('-', '_')
        if 'project_src' in b:
            return z
    return None

zips = all_zips()
print('.zip files:', zips or '(none — OK if Kaggle extracted folders)')

Z_DATA = MANUAL_DATA
Z_SRC = MANUAL_SRC
if not Z_DATA or not Z_SRC:
    Z_DATA = Z_DATA or pick_data_zip(zips)
    Z_SRC = Z_SRC or pick_src_zip(zips)
if not Z_DATA or not Z_SRC:
    Z_DATA = Z_DATA or find_named_dir(ROOT, 'pids_bench_v3')
    Z_SRC = Z_SRC or find_named_dir(ROOT, 'project_src')

if not Z_DATA or not Z_SRC:
    raise RuntimeError(
        'Could not find pids_bench_v3 and project_src (as .zip or folders under /kaggle/input).\n'
        'Attach your dataset to THIS notebook (Add data), Save Version, re-run.\n'
        'Or set MANUAL_DATA / MANUAL_SRC to the paths shown in the Input tab.\n'
        f'Got DATA={Z_DATA!r} SRC={Z_SRC!r}'
    )

print('\nUsing:')
print('  DATA', Z_DATA, '(zip)' if Z_DATA.endswith('.zip') else '(folder)')
print('  SRC ', Z_SRC, '(zip)' if Z_SRC.endswith('.zip') else '(folder)')

In [ ]:
# ── Cell 2: Dependencies (PyTorch must support your GPU) ─────────────────
# Default Kaggle torch = CUDA 12.x builds that only support sm_70+ (T4, A100, …).
# Tesla P100 = sm_60 → use cu118 wheels. If you have T4, you can skip the cu118 lines.
import subprocess, sys

def sh(cmd):
    subprocess.check_call(cmd, shell=True)

sh(f"{sys.executable} -m pip uninstall -y torch torchvision torchaudio 2>/dev/null || true")
sh(
    f"{sys.executable} -m pip install -q torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu118"
)
sh(
    f"{sys.executable} -m pip install -q -U transformers datasets scikit-learn evaluate accelerate"
)
# Transformers may pull a newer torch; pin back to cu118 for P100
sh(
    f"{sys.executable} -m pip install -q torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu118"
)

print("Installed. Run the next cell to verify CUDA + GPU name.")

In [ ]:
# ── Cell 3: GPU check (run after Cell 2) ─────────────────────────────────
import torch
print("torch:", torch.__version__)
print("CUDA built:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.randn(1, device="cuda")
    print("smoke OK:", x.device)

In [ ]:
# ── Cell 4: Build project tree in /kaggle/working/pids_project ────────────
import zipfile, os, shutil

PROJECT_DIR = '/kaggle/working/pids_project'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

def merge_dir_into(src_root, dst_root):
    os.makedirs(dst_root, exist_ok=True)
    for name in os.listdir(src_root):
        s = os.path.join(src_root, name)
        d = os.path.join(dst_root, name)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)

def load_path(path, dst_root):
    if path.lower().endswith('.zip'):
        with zipfile.ZipFile(path, 'r') as z:
            z.extractall(dst_root)
    elif os.path.isdir(path):
        merge_dir_into(path, dst_root)
    else:
        raise FileNotFoundError(path)

load_path(Z_DATA, PROJECT_DIR)
load_path(Z_SRC, PROJECT_DIR)

# If the bench ended up as .../pids_bench_v3/train.csv instead of data/pids_bench_v3/
train_std = os.path.join(PROJECT_DIR, 'data', 'pids_bench_v3', 'train.csv')
if not os.path.isfile(train_std):
    alt = os.path.join(PROJECT_DIR, 'pids_bench_v3', 'train.csv')
    if os.path.isfile(alt):
        os.makedirs(os.path.join(PROJECT_DIR, 'data'), exist_ok=True)
        shutil.move(
            os.path.join(PROJECT_DIR, 'pids_bench_v3'),
            os.path.join(PROJECT_DIR, 'data', 'pids_bench_v3'),
        )

assert os.path.exists(os.path.join(PROJECT_DIR, 'data', 'pids_bench_v3', 'train.csv')), \
    'Expected data/pids_bench_v3/train.csv — check your zip/folder layout.'
assert os.path.exists(os.path.join(PROJECT_DIR, 'src', 'baselines', 'deberta_v3.py')), \
    'Expected src/baselines/deberta_v3.py — check project_src contents.'
print('OK →', PROJECT_DIR)

In [ ]:
# ── Cell 5: Working directory ────────────────────────────────────────────
import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(os.getcwd())

In [ ]:
# ── Cell 6: Train DeBERTa (T4/P100: batch 8, grad_accum 2) ────────────────
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from src.baselines.deberta_v3 import run_train as deberta_train

deberta_train(
    num_epochs=3,
    batch_size=8,
    gradient_accumulation_steps=2,
    lr=2e-5,
    seed=42,
)
print('DONE')

In [ ]:
# ── Cell 7: Show summary + copy to /kaggle/working for download ──────────
import json, shutil
src = 'outputs/deberta_v3/summary.json'
with open(src) as f:
    s = json.load(f)
print(json.dumps(s, indent=2)[:4000])
shutil.copy(src, '/kaggle/working/deberta_summary.json')
print('\nAlso saved to /kaggle/working/deberta_summary.json — use Output tab to download.')